**Simulación del movimiento de células en contraflujo**

**Importación de librerias necesarias**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import ScalarFormatter

%matplotlib inline
from IPython.display import HTML

from scipy.optimize import minimize, dual_annealing

: 

**Definición de parámetros necesarios**

In [ ]:
#Variables fijas
num_celulas = 50
radio = 0.5
dt = 0.05
sigma_pos = 0
m = 1 #Puede ser 1 o 2

#Variables graficas
limx =10
limy=10

#Variables de analisis (las que serán modificadas más adelante)
sigma_angulo = 1
w = 1
u0 = 1
V0 = 1

#Para el posterior analisis, es necesario establecer un semilla especifica para los números aleatorios.
#Si no se van a realizar los analisis de sensibilidad y optimización, se debe eliminar esta línea.
np.random.seed(20) 

#Se definen los angulos y posiciones iniciales
angulos = np.random.rand(num_celulas) *2*np.pi
alineacion = np.zeros(num_celulas)
pos_x = np.random.rand(num_celulas) *limx
pos_y = np.random.rand(num_celulas) *limy
fuerza_x = np.zeros(num_celulas)
fuerza_y = np.zeros(num_celulas)

#Variables para animar
frame_grabar = 400  #Cuantos frames se van a analizar (se usa un valor pequeño para agilizar procesos posteriores)
frames_total = 500 #Se usan más frames que los analizados para asegurar el analisis en el regimen estacionario
frames_estabilizar = frames_total - frame_grabar
posx_real = np.copy(pos_x)
posy_real = np.copy(pos_y)
vel_x = np.zeros(num_celulas)
vel_y = np.zeros(num_celulas)

#Las matrices para guardar los datos 
historial_posx = np.zeros((frame_grabar, num_celulas))
historial_posy = np.zeros((frame_grabar, num_celulas))
historial_velx = np.zeros((frame_grabar, num_celulas))
historial_vely = np.zeros((frame_grabar, num_celulas))

**Definición de funciones necesarias para la el cálculo de posición**

Cálculo de la fuerza:

In [ ]:
def fuerzas(): 
    global pos_x, pos_y, fuerza_x, fuerza_y, alineacion
    fuerza_x.fill(0)
    fuerza_y.fill(0)
    alineacion.fill(0)

    size_grilla = 2*radio #Elegimos este ya que es la distancia maxima de interaccion de la formula de fuerza mas adelante

    #Lo siguente es para que las celulas se puedan detectar si estan en los bordes
    columnas = int(limx/ size_grilla)
    filas = int(limy/size_grilla)

    for i in range(num_celulas):
        grid_x1 = int(pos_x[i]/size_grilla)
        grid_y1 = int(pos_y[i]/size_grilla)
        for j in range( num_celulas):
            if i != j:
                grid_x2 = int(pos_x[j]/size_grilla)
                grid_y2 = int(pos_y[j]/size_grilla)

                dist_grillax = abs(grid_x1-grid_x2)
                dist_grillay = abs(grid_y1-grid_y2)

            #Para ver si estan cerca, pero en distinto borde
                if dist_grillax > columnas / 2:
                    dist_grillax = columnas - dist_grillax
                if dist_grillay > filas / 2:
                    dist_grillay = filas - dist_grillay

                if dist_grillax > 1 or dist_grillay > 1:  #Si es mayor a uno es que estan a mas de una grilla de dif
                    continue

                dx = pos_x[i] - pos_x[j]
                dy = pos_y[i] - pos_y[j]

            #Para ver si el camino mas corto es pasando por los bordes 
                if dx > limx / 2:
                    dx -= limx
                elif dx < -limx / 2:
                    dx += limx
                
                if dy > limy / 2:
                    dy -= limy
                elif dy < -limy / 2:
                    dy += limy

                distancia = np.sqrt(dx**2 + dy**2)
                if distancia < 2*radio and distancia > 0:
                    fuerza_x[i] += (2*radio-distancia)*dx/distancia
                    fuerza_y[i] += (2*radio-distancia)*dy/distancia
            
                    alineacion[i] += w*np.sin(m*(angulos[j]-angulos[i]))

Actualización del ángulo:

In [ ]:
def actu_angulo():
    global angulos
    for i in range(num_celulas):
        gauss_angulo =  np.random.normal(0,1)
        angulos[i] = angulos[i] + (alineacion[i]*dt) + sigma_angulo*gauss_angulo*np.sqrt(dt) 
    angulos = angulos % (2 * np.pi)

Cálculo de la nueva posición:

In [ ]:
def actu_pos_angulo():
    global pos_x, pos_y, posx_real, posy_real, vel_x, vel_y
    for i in range(num_celulas):
        gauss_x =  np.random.normal(0,1)
        gauss_y =  np.random.normal(0,1)

        direc_x = np.cos(angulos[i])
        direc_y = np.sin(angulos[i])
        #Se separa el calculo de la posicion par aguardar todos los datos necesarios
        dx = u0*dt*fuerza_x[i] + sigma_pos*np.sqrt(dt)*gauss_x + V0*direc_x*dt
        dy = u0*dt*fuerza_y[i] + sigma_pos*np.sqrt(dt)*gauss_y + V0*direc_y*dt
        vel_x[i] = dx/dt
        vel_y[i] = dy/dt
        posx_real[i] = posx_real[i] + dx
        posy_real[i] = posy_real[i] + dy
        #Ahora si calculamos la nuva posicion
        pos_x[i] = pos_x[i] + dx
        pos_y[i] = pos_y[i] + dy

Aplicación de condiciones de borde (las células aparecen por el lado opuesto al salir de los límites):

In [ ]:
def aplicar_bordes():
    global pos_x, pos_y
    pos_x = pos_x % limx
    pos_y = pos_y % limy

**Animación del movimiento**

Se estabilizan las células:

In [ ]:
print("Preparando régimen estacionario")
for i in range(frames_estabilizar):
    fuerzas()
    actu_angulo()
    actu_pos_angulo()
    aplicar_bordes()
print("Comienza animacion")

Se anima el movimiento de las células:

In [ ]:
fig, ax = plt.subplots(figsize= (6,6))
ax.set_xlim(0, limx)
ax.set_ylim(0, limy)

grupo_celulas, = ax.plot([], [], 'bo', markersize=20) #Markersize es el tamano de las celulas


def update(frame):
    global historial_posx, historial_posy, historial_velx, historial_vely
    fuerzas()
    actu_angulo()
    actu_pos_angulo()
    aplicar_bordes()
    historial_posx[frame, :] = posx_real
    historial_posy[frame, :] = posy_real
    historial_velx[frame, :] = vel_x
    historial_vely[frame, :] = vel_y
    grupo_celulas.set_data(pos_x,pos_y)
    return grupo_celulas,

#Para ver la grilla, si no se quiere hay que comentar todo esta parte
size_grilla = 2*radio
ax.set_xticks(np.arange(0, limx + 1, size_grilla))
ax.set_yticks(np.arange(0, limy + 1, size_grilla))
# Activamos la grilla 
ax.grid(True, linestyle='--', color='gray', alpha=0.5)

#Ir cambiando interval para velocidad de la animacion
animacion = FuncAnimation(fig, update, frames=frame_grabar, blit=True, interval=20, repeat = False)
plt.close()
HTML(animacion.to_jshtml())

**Análisis de las variables**

Función para MSD:

In [ ]:
def calcular_msd(hist_posx,hist_posy):
    T = frame_grabar
    msd = np.zeros(T)
    for k in range (1,T):
        suma_msd = 0
        for t in range(T-k):
            for i in range(num_celulas):
                dx = hist_posx[t+k,i] - hist_posx[t,i]
                dy = hist_posy[t+k,i] - hist_posy[t,i]
                suma_msd += (dx**2) + (dy**2)
        msd[k] = (1/(num_celulas*(T-k)))*suma_msd
    return msd

Función para distribución de velocidades:

In [ ]:
def graf_vel(hist_vx,hist_vy):
    vx_plano = hist_vx.flatten()
    vy_plano = hist_vy.flatten()
    return vx_plano, vy_plano

Función para autocorrelación:

In [ ]:
def calcular_autocorrelacion(hist_v):
    T = frame_grabar
    auto_cor = np.zeros(T)
    var_v = np.sum(hist_v**2) / (num_celulas*T)
    for k in range(T):
        suma_auto = 0
        for t in range(T-k):
            for i in range (num_celulas):
                vt = hist_v[t,i]
                vtk = hist_v[t+k,i]
                suma_auto += vt*vtk
        auto_cor[k] = ((1/(num_celulas*(T-k)))*suma_auto) / var_v
    return auto_cor

Función para correlación y distribución de pares:

In [ ]:
def calcular_correlacion_y_pares(hist_posx,hist_posy,hist_velx,hist_vely):
    #La funcion esta preparada para un sistema no cuadrado, se puede simplificar bastante asumiendo que siempre sera cuadrado
    T = frame_grabar
    N = num_celulas
    posx_lim = hist_posx % limx
    posy_lim = hist_posy % limy
    bins = 50 #Cuantos anillos van a ser, ir cambiando si se ve mal el resultado 
    rmax = min(limx,limy)/2
    dr = rmax / bins
    radios = np.linspace(dr/2, rmax - dr/2, bins) #esto es para graficar, poniendo los puntos en la mitad del anillo
    numerador_x = np.zeros(bins)
    numerador_y = np.zeros(bins)
    denominador = np.zeros(bins)
    for t in range(T):
        velx = hist_velx[t]
        vely = hist_vely[t]
        posx = posx_lim[t]
        posy = posy_lim[t]
        for i in range(N):
            for j in range(N):
                if i != j:
                    dx = posx[i] - posx[j]
                    dy = posy[i] - posy[j]
                    if dx > limx / 2: 
                        dx -= limx
                    elif dx < -limx / 2: 
                        dx += limx
                    if dy > limy / 2: 
                        dy -= limy
                    elif dy < -limy / 2: 
                        dy += limy
                    r = np.sqrt(dx**2 + dy**2)
                    if r < rmax:
                        b = int(r/dr) #Ve en que anillo esta 
                        if b < bins:
                            numerador_x[b] += velx[i]*velx[j]
                            numerador_y[b] += vely[i]*vely[j]
                            denominador[b] += 1
    cxx = np.zeros(bins)
    cyy = np.zeros(bins)
    gr = np.zeros(bins)
    densidad = N/ (limx*limy)
    for b in range (bins):
        if denominador[b] > 0:
            cxx[b] = numerador_x[b] / denominador[b]
            cyy[b] = numerador_y[b] / denominador[b]
        #Aqui se hace la distribucion de pares
        r_actual = radios[b]
        area = 2*np.pi*r_actual*dr
        gr[b] = denominador[b] / (T*N*densidad*area)
    #Aqui se hace la normalizacion
    cxx0 = np.sum(hist_velx**2) / (N*T)
    cyy0 = np.sum(hist_vely**2) / (N*T)
    cxx_norm = cxx/cxx0
    cyy_norm = cyy/cyy0

    return radios, cxx_norm, cyy_norm, gr

Creación de los gráficos:

In [ ]:
msd_resultado = calcular_msd(historial_posx, historial_posy)
histograma_vx, histograma_vy = graf_vel(historial_velx, historial_vely)
auto_cor_vlx = calcular_autocorrelacion(historial_velx)
auto_cor_vly = calcular_autocorrelacion(historial_vely)
radios, cxx_r, cyy_r, g_r = calcular_correlacion_y_pares(historial_posx, historial_posy, historial_velx, historial_vely)


fig_stats = plt.figure(figsize=(18, 10))
# Se crea la grilla invisible de 2 filas y 6 columnas
gs = GridSpec(2, 6, figure=fig_stats)

# Se asignan los espacios a cada gráfico
ax1 = fig_stats.add_subplot(gs[0, 0:2]) # Arriba: Izquierda
ax2 = fig_stats.add_subplot(gs[0, 2:4]) # Arriba: Centro
ax3 = fig_stats.add_subplot(gs[0, 4:6]) # Arriba: Derecha

ax4 = fig_stats.add_subplot(gs[1, 1:3]) # Abajo: Centrado entre 1 y 2
ax5 = fig_stats.add_subplot(gs[1, 3:5]) # Abajo: Centrado entre 2 y 3

tiempo_tau = np.arange(frame_grabar) * dt 
mitad_T = int(frame_grabar / 2)



# Gráfico 1: MSD
ax1.plot(tiempo_tau, msd_resultado, lw=2)
ax1.set_title('Desplazamiento Cuadrático Medio (MSD)')
ax1.set_xlabel('Tiempo ($\\tau$) [segundos]')
ax1.set_ylabel('$MSD(\\tau)$')
ax1.grid(True, linestyle='--', alpha=0.5)

# Gráfico 2: Distribución de Velocidades
ax2.hist(histograma_vx, bins=30, density=True, alpha=0.6, color='red', label='$P(v_x)$')
ax2.hist(histograma_vy, bins=30, density=True, alpha=0.6, color='orange', label='$P(v_y)$')
ax2.set_title('Distribución de Velocidades')
ax2.set_xlabel('Velocidad')
ax2.set_ylabel('Probabilidad')
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.5)

# Gráfico 3: Autocorrelación Temporal
ax3.plot(tiempo_tau[:mitad_T], auto_cor_vlx[:mitad_T], color='red', label='$\\tilde{C}_{xx}(\\tau)$')
ax3.plot(tiempo_tau[:mitad_T], auto_cor_vly[:mitad_T], color='orange', label='$\\tilde{C}_{yy}(\\tau)$')
ax3.axhline(0, color='black', linestyle='--')
ax3.set_title('Autocorrelación Temporal de Velocidades')
ax3.set_xlabel('Tiempo ($\\tau$) [segundos]')
ax3.set_ylabel('Correlación Normalizada')
ax3.legend()
ax3.grid(True, linestyle='--', alpha=0.5)

# Gráfico 4: Correlación Espacial
ax4.plot(radios, cxx_r, color='red', marker='.', label='$\\tilde{C}_{xx}(r)$')
ax4.plot(radios, cyy_r, color='orange', marker='.', label='$\\tilde{C}_{yy}(r)$')
ax4.axhline(0, color='black', linestyle='--')
ax4.set_title('Correlación Espacial de Velocidades')
ax4.set_xlabel('Distancia radial ($r$)')
ax4.set_ylabel('Correlación Normalizada')
ax4.legend()
ax4.grid(True, linestyle='--', alpha=0.5)

# Gráfico 5: Función de Distribución de Pares g(r)
ax5.plot(radios, g_r, color='green', lw=2)
ax5.axhline(1, color='black', linestyle='--', label='Homogéneo ($g(r)=1$)')
ax5.set_title('Función de Distribución de Pares $g(r)$')
ax5.set_xlabel('Distancia radial ($r$)')
ax5.set_ylabel('$g(r)$')
ax5.legend()
ax5.grid(True, linestyle='--', alpha=0.5)


plt.tight_layout()
plt.show()



**Análisis de sensibilidad**

Se crea una base para el análisis de sensibilidad. Para esto, se debe hacer el cálculo nuevamente de las posiciones, pues al gráficar estos se ven levemente alterados.

In [ ]:
#Se reduce el numero de estas variables para agilizar el codigo
num_celulas = 25
frame_grabar = 50
frames_total = 100
frames_estabilizar = frames_total - frame_grabar
np.random.seed(20)

angulos = np.random.rand(num_celulas) *2*np.pi
pos_x = np.random.rand(num_celulas) *limx
pos_y = np.random.rand(num_celulas) *limy
posx_real = np.copy(pos_x)
posy_real = np.copy(pos_y)
vel_x = np.zeros(num_celulas)
vel_y = np.zeros(num_celulas)

# Estabilización Pura
for i in range(frames_estabilizar):
    fuerzas()
    actu_angulo()
    actu_pos_angulo()
    aplicar_bordes()

hist_posx_orig = np.zeros((frame_grabar, num_celulas))
hist_posy_orig = np.zeros((frame_grabar, num_celulas))
hist_velx_orig = np.zeros((frame_grabar, num_celulas))
hist_vely_orig = np.zeros((frame_grabar, num_celulas))

# Grabación Pura
for frame in range (frame_grabar):
    fuerzas()
    actu_angulo()
    actu_pos_angulo()
    aplicar_bordes()
    hist_posx_orig[frame, :] = posx_real
    hist_posy_orig[frame, :] = posy_real
    hist_velx_orig[frame, :] = vel_x
    hist_vely_orig[frame, :] = vel_y


msd_orig = calcular_msd(hist_posx_orig, hist_posy_orig)
auto_x_orig = calcular_autocorrelacion(hist_velx_orig)
auto_y_orig = calcular_autocorrelacion(hist_vely_orig)
radios, cxx_orig, cyy_orig, gr_orig = calcular_correlacion_y_pares(hist_posx_orig, hist_posy_orig, hist_velx_orig, hist_vely_orig)

auto_prom_orig = (auto_x_orig + auto_y_orig) / 2
c_prom_orig = (cxx_orig + cyy_orig) / 2

#Verdadero vector de referencia, con todas las variables iguales a 1 
vector_original = np.concatenate([msd_orig, auto_prom_orig, c_prom_orig, gr_orig])

Los vectores no poseen igual longitud. Por ello, se debe considerar un peso especifico de cada indicador para el cálculo del error.

In [ ]:
#Escalas para mejorar el error 
escala_msd = np.max(msd_orig) 
escala_autocor = 1
escala_cor = 1
escala_g = np.max(gr_orig)

peso_msd = np.ones(len(msd_orig))/(escala_msd**2)
peso_autocor = np.ones(len(auto_prom_orig))/(escala_autocor**2)
peso_cor = np.ones(len(c_prom_orig))/(escala_cor**2)
peso_g = np.ones(len(gr_orig))/(escala_g**2)
vector_peso = np.concatenate([peso_msd,peso_autocor,peso_cor,peso_g])

Se define la ecuación para calcular el error:

In [ ]:
def calcular_error(prueba_sigma_angulo, prueba_w, prueba_u0, prueba_V0, vector_original):
    global sigma_angulo, w, u0, V0
    global pos_x, pos_y, posx_real, posy_real, angulos, vel_x, vel_y, vector_peso
    sigma_angulo = prueba_sigma_angulo
    w = prueba_w
    u0 = prueba_u0
    V0 = prueba_V0

    #Debemos fijar una seed en el random para que simplex no se maree. Tome un valor arbitrario
    np.random.seed(20)

    #Se reinician los datos 
    angulos = np.random.rand(num_celulas) *2*np.pi
    pos_x = np.random.rand(num_celulas) *limx
    pos_y = np.random.rand(num_celulas) *limy
    posx_real = np.copy(pos_x)
    posy_real = np.copy(pos_y)
    vel_x = np.zeros(num_celulas)
    vel_y = np.zeros(num_celulas)

    #Se prepara
    for i in range(frames_estabilizar):
        fuerzas()
        actu_angulo()
        actu_pos_angulo()
        aplicar_bordes()

    historial_posx = np.zeros((frame_grabar, num_celulas))
    historial_posy = np.zeros((frame_grabar, num_celulas))
    historial_velx = np.zeros((frame_grabar, num_celulas))
    historial_vely = np.zeros((frame_grabar, num_celulas))
    #Se graban los datos
    for frame in range (frame_grabar):
        fuerzas()
        actu_angulo()
        actu_pos_angulo()
        aplicar_bordes()
        historial_posx[frame, :] = posx_real
        historial_posy[frame, :] = posy_real
        historial_velx[frame, :] = vel_x
        historial_vely[frame, :] = vel_y
    msd_prueba = calcular_msd(historial_posx, historial_posy)
    auto_cor_vlx = calcular_autocorrelacion(historial_velx)
    auto_cor_vly = calcular_autocorrelacion(historial_vely)
    radios, cxx_r, cyy_r, g_r_prueba = calcular_correlacion_y_pares(historial_posx, historial_posy, historial_velx, historial_vely)
    auto_cor_prom_prueba = (auto_cor_vlx+auto_cor_vly)/2
    c_prom_prueba = (cxx_r + cyy_r)/2
    vector_prueba = np.concatenate([msd_prueba,auto_cor_prom_prueba,c_prom_prueba,g_r_prueba])
    #Se calcula el error
    dif_2 = (vector_prueba-vector_original)**2
    error = np.sum(dif_2*vector_peso)
    return error

Se cálcula la sensibilidad, y se gráfica:

In [ ]:
valores_prueba = np.linspace(0,2,51)

errores_angulo_sigma = []
errores_w=[]
errores_u0 = []
errores_V0 = []

for val in valores_prueba:
    errores_angulo_sigma.append(calcular_error(val,1,1,1,vector_original))
    errores_w.append(calcular_error(1,val,1,1,vector_original))
    errores_u0.append(calcular_error(1,1,val,1,vector_original))
    errores_V0.append(calcular_error(1,1,1,val,vector_original))


formateador = ScalarFormatter()
formateador.set_scientific(False)
fig_sens, ax = plt.subplots(figsize=(10, 6))


ax.plot(valores_prueba, errores_angulo_sigma, marker='s', label='Sensibilidad a $\\sigma$ (Ruido)')
ax.plot(valores_prueba, errores_w, marker='o', label='Sensibilidad a $w$ (Alineación)')
ax.plot(valores_prueba, errores_u0, marker='d', label='Sensibilidad a $u_0$ (Repulsión)')
ax.plot(valores_prueba, errores_V0, marker='^', label='Sensibilidad a $V_0$ (Velocidad)')

# Se marca el valor objetivo central
ax.axvline(1.0, color='black', linestyle='--', label='Valor Objetivo (1.0)')

# Formato general del gráfico
ax.set_title('Sensibilidad OAT (Intervalo 0.0 a 2.0)', fontsize=14)
ax.set_xlabel('Valor del Parámetro', fontsize=12)
ax.set_ylabel('Error Relativo Total', fontsize=12)

# Mantenemos la escala logarítmica por las grandes variaciones de error
ax.set_yscale('log')

# Aplicamos el formateador para evitar que salgan números como 10^1
formateador = ScalarFormatter()
formateador.set_scientific(False)
ax.yaxis.set_major_formatter(formateador)

# Cuadrícula y leyenda
ax.grid(True, which="both", ls="--", alpha=0.5)
ax.legend()

plt.tight_layout()
plt.show()


**Optimización**

**1. Downhill Simplex Method in Multidimensions**

Se definen las funciones utilizadas en el método:

In [ ]:
#p es la matriz 5X4 que tiene los vertices de prueba (5 opciones para mis variables)
#y es un vector de 5, que posee el error asociado a cada fila de p

def amotry(vertices_p,resultados_y,psum,funcion,peor_vert,factor):
    n_dim = vertices_p.shape[1]
    factor1 = (1-factor)/n_dim
    factor2 = factor1 - factor
    vertice_try = psum*factor1 - vertices_p[peor_vert]*factor2  #De esta forma no es necesario el for #vertice_try es un vector
    resultado_try = funcion(vertice_try)  #resultados_try es un valor
    if resultado_try < resultados_y[peor_vert]:
        resultados_y[peor_vert] = resultado_try
        psum += vertice_try - vertices_p[peor_vert]
        vertices_p[peor_vert] = vertice_try
    return resultado_try

def amoeba(vertices_p, resultados_y,ftol,funcion,max_iter=1000):
    TINY = 1e-10
    mpts = vertices_p.shape[0]
    psum = np.sum(vertices_p,axis=0)  #Es un vector, donde cada columna es la suma de esa variable
    for i in range(max_iter):
        peor_vert = np.argmax(resultados_y)  #ihi
        mejor_vert = np.argmin(resultados_y)  #ilo
        #Nos falta el segundo peor, para eso podemos elimiar el peor y encontrar el nuevo peor
        copia_y = np.copy(resultados_y)
        copia_y[peor_vert] = -np.inf  #No se puede eliminar, porque el indice obtenido no va a coincidir en el vector original
        peor2_vert = np.argmax(copia_y)  #inhi
        rtol = 2*abs(resultados_y[peor_vert]-resultados_y[mejor_vert])/(abs(resultados_y[peor_vert])+abs(resultados_y[mejor_vert])+TINY)
        if rtol < ftol:
            print(f"Se alcanzo la conevregencia en {i} iteraciones")
            return vertices_p[mejor_vert],resultados_y[mejor_vert], i #Se devuelve el mejor resultado y su error
        #Aqui inicia el movimiento de la figura
        # -1 refleja el peor punto al otro lado 
        resultado_ytry = amotry(vertices_p,resultados_y,psum,funcion,peor_vert,-1)
        if resultado_ytry <= resultados_y[mejor_vert]:
            #La refexion da un buen resultado, al ser mejor que el actual mejor. Se usa 2 para aumentar la distancia en esa direccion
            resultado_ytry = amotry(vertices_p,resultados_y,psum,funcion,peor_vert,2)
        elif resultado_ytry >= resultados_y[peor2_vert]:
            #Si la reflexion no dio un mejor resultado, se hace una contraccion en la direccion con 0.5
            y_save = resultados_y[peor_vert]
            resultado_ytry = amotry(vertices_p,resultados_y,psum,funcion,peor_vert,0.5)
            if resultado_ytry >= y_save:
                #Si aun se tiene un peor resultado, se encoje toda la figura usando de referencia el mejor vertice
                for j in range(mpts):
                    if j != mejor_vert:
                        vertices_p[j] = 0.5 * (vertices_p[j]+vertices_p[mejor_vert])
                        resultados_y[j] = funcion(vertices_p[j])
                psum = np.sum(vertices_p, axis = 0)
    print ("Se alcanzo el numero maximo de iteraciones")
    return vertices_p[mejor_vert],resultados_y[mejor_vert], max_iter

#Por como esta escrito calcular_error(), hay que hacer una arreglo 
def funcion_simplex(arreglo_parametros):
    sim_sigma, sim_w, sim_u0, sim_v0 = arreglo_parametros
    return calcular_error(sim_sigma, sim_w,sim_u0, sim_v0, vector_original)

Se ejecuta simplex. Para un mejor análisis, el metodo se ejecuta multiples veces, cambiando el punto inicial y la tolerancia.

In [ ]:
tolerancias = [1e-6,1e-7,1e-8,1e-9]
puntos_iniciales = [np.array([1.5,1.5,1.5,1.5]) ,np.array([2,2,2,2]),np.array([3,3,3,3])]
etiquetas_print = ["Punto inicial 1.5","Punto inicial 2","Punto inicial 3"]

matriz_iteraciones_simplex = np.zeros((len(puntos_iniciales), len(tolerancias)))
matriz_errores_simplex = np.zeros((len(puntos_iniciales), len(tolerancias)))

ndim = 4
matriz_resultados_simplex =np.zeros((len(puntos_iniciales), len(tolerancias), ndim))

print("\n" + "="*50)
print("INICIANDO SIMPLEX")
print("="*50)

for i in range(len(puntos_iniciales)):
    p_inicial = puntos_iniciales[i]
    print(f"Evaluamos el punto inicial {p_inicial}")
    for j in range(len(tolerancias)):
        tolerancia = tolerancias[j]
        print(f"Usamos la toleracia {tolerancia}")
        ndim = len(p_inicial)
        mpts = ndim + 1
        vertices_p = np.zeros((mpts,ndim))  #p
        vertices_p[0] = p_inicial
        paso = 0.05
        for k in range(1,mpts):
                vertices_p[k] = np.copy(p_inicial)
                vertices_p[k,k-1] +=paso
        resultados_y = np.zeros(mpts)  #y
        for k in range(mpts):
            resultados_y[k] = funcion_simplex(vertices_p[k])
        mejores_p, menor_err, iteraciones_tot = amoeba(vertices_p,resultados_y,tolerancia, funcion_simplex)
        matriz_iteraciones_simplex[i,j] = iteraciones_tot
        matriz_errores_simplex[i,j] = menor_err
        matriz_resultados_simplex[i,j] = mejores_p
        print(f"Terminado en {iteraciones_tot} iteraciones, con un error de {menor_err}. Se llego al punto {mejores_p}")

print("\nSimplex Completado")

Se grafican los resutados de simplex.

In [ ]:
fig_comp, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
colores = ['#1f77b4', '#ff7f0e', '#2ca02c']
marcadores = ['o', 's', '^']

for i in range(len(puntos_iniciales)):
    # Gráfico de Iteraciones
    ax1.plot(tolerancias, matriz_iteraciones_simplex[i], color=colores[i], marker=marcadores[i], linestyle='-', linewidth=2, markersize=8, label=etiquetas_print[i])
    
    # Gráfico de Error Final
    ax2.plot(tolerancias, matriz_errores_simplex[i], color=colores[i], marker=marcadores[i], linestyle='-', linewidth=2, markersize=8, label=etiquetas_print[i])

# Formato Gráfico 1: Iteraciones vs Tolerancia
ax1.set_title('Convergencia: Iteraciones requeridas', fontsize=14)
ax1.set_xlabel('Tolerancia ($ftol$)', fontsize=12)
ax1.set_ylabel('Número de Iteraciones', fontsize=12)
ax1.set_xscale('log')
ax1.grid(True, which="both", ls="--", alpha=0.5)
ax1.legend()

# Formato Gráfico 2: Error vs Tolerancia
ax2.set_title('Precisión: Error Relativo Alcanzado', fontsize=14)
ax2.set_xlabel('Tolerancia ($ftol$)', fontsize=12)
ax2.set_ylabel('Error Relativo Mínimo', fontsize=12)
ax2.set_xscale('log')
ax2.set_yscale('log') # Eje Y logarítmico para el error


formateador = ScalarFormatter()
formateador.set_scientific(False)
ax2.yaxis.set_major_formatter(formateador)

ax2.grid(True, which="both", ls="--", alpha=0.5)
ax2.legend()

plt.tight_layout()
plt.show()


**2. Continuous Minimization by Simulated Annealing**

Se crean las funciones necesarias para este método: 

In [ ]:
#ter abrevia termico
def amotry_annealing(vertices_p,resultados_y,psum,funcion,peor_vert,factor,T):
    n_dim = vertices_p.shape[1]
    factor1 = (1-factor)/n_dim
    factor2 = factor1 - factor
    vertice_try = psum*factor1 - vertices_p[peor_vert]*factor2  #De esta forma no es necesario el for #vertice_try es un vector
    resultado_try = funcion(vertice_try)  #resultados_try es un valor
    #Parte nueva de annealing
    fluctuacion = -T*np.log(np.random.rand()) #El termino termina siendo positivo
    resultado_try_ter = resultado_try - fluctuacion
    if resultado_try_ter < resultados_y[peor_vert]:
        resultados_y[peor_vert] = resultado_try
        psum += vertice_try - vertices_p[peor_vert]
        vertices_p[peor_vert] = vertice_try
    return resultado_try_ter

def amoeba_annealing(vertices_p, resultados_y,ftol,funcion,T_inicial,factor_enfriamiento,max_iter=1000):
    TINY = 1e-10
    mpts = vertices_p.shape[0]
    psum = np.sum(vertices_p,axis=0)  #Es un vector, donde cada columna es la suma de esa variable

    #Variables nuvas que se deben definir 
    T = T_inicial
    mejor_error_hist = np.min(resultados_y)
    mejor_vert_hist = np.copy(vertices_p[np.argmin(resultados_y)])

    for i in range(max_iter):
        y_ter = np.copy(resultados_y)
        for j in range (mpts):

            fluctuaciones = -T * np.log(np.random.rand())
            y_ter[j] += fluctuaciones

        peor_vert = np.argmax(y_ter)  #ihi
        mejor_vert = np.argmin(y_ter)  #ilo
        #Nos falta el segundo peor, para eso podemos elimiar el peor y encontrar el nuevo peor
        copia_y_ter = np.copy(y_ter)
        copia_y_ter[peor_vert] = -np.inf  #No se puede eliminar, porque el indice obtenido no va a coincidir en el vector original
        peor2_vert = np.argmax(copia_y_ter)  #inhi

        #Se va actualizando el mejor punto, por si llega a quedarse atrapado en un minimo local al saltar por T
        if resultados_y[mejor_vert] < mejor_error_hist:
            mejor_error_hist = resultados_y[mejor_vert]
            mejor_vert_hist = np.copy(vertices_p[mejor_vert])

        rtol = 2*abs(resultados_y[peor_vert]-resultados_y[mejor_vert])/(abs(resultados_y[peor_vert])+abs(resultados_y[mejor_vert])+TINY)
        if rtol < ftol and T < 1e-4:
            print(f"Se alcanzo la conevregencia en {i} iteraciones (T={T})")
            return mejor_vert_hist,mejor_error_hist, i #Se devuelve el mejor resultado y su error
        #Aqui inicia el movimiento de la figura
        # -1 refleja el peor punto al otro lado 
        resultado_ytry_ter= amotry_annealing(vertices_p,resultados_y,psum,funcion,peor_vert,-1,T)
        if resultado_ytry_ter <= y_ter[mejor_vert]:
            #La refexion da un buen resultado, al ser mejor que el actual mejor. Se usa 2 para aumentar la distancia en esa direccion
            resultado_ytry_ter = amotry_annealing(vertices_p,resultados_y,psum,funcion,peor_vert,2,T)
        elif resultado_ytry_ter >= y_ter[peor2_vert]:
            #Si la reflexion no dio un mejor resultado, se hace una contraccion en la direccion con 0.5
            y_save = y_ter[peor_vert]
            resultado_ytry_ter = amotry_annealing(vertices_p,resultados_y,psum,funcion,peor_vert,0.5,T)
            if resultado_ytry_ter >= y_save:
                #Si aun se tiene un peor resultado, se encoje toda la figura usando de referencia el mejor vertice
                for j in range(mpts):
                    if j != mejor_vert:
                        vertices_p[j] = 0.5 * (vertices_p[j]+vertices_p[mejor_vert])
                        resultados_y[j] = funcion(vertices_p[j])
                psum = np.sum(vertices_p, axis = 0)
        T = T * factor_enfriamiento
    print ("Se alcanzo el numero maximo de iteraciones")
    return mejor_vert_hist, mejor_error_hist, max_iter

Se ejecuta annealing. Para poder hacer la comparación más adenlante, se utilizan los mismos valores iniciales y tolerancias.

In [ ]:
tolerancias = [1e-6, 1e-7, 1e-8, 1e-9]
puntos_iniciales = [np.array([1.5,1.5,1.5,1.5]) ,np.array([2,2,2,2]),np.array([3,3,3,3])]
etiquetas_print = ["Punto inicial 1.5","Punto inicial 2","Punto inicial 3"]

matriz_iteraciones_annealing = np.zeros((len(puntos_iniciales), len(tolerancias)))
matriz_errores_annealing = np.zeros((len(puntos_iniciales), len(tolerancias)))

ndim = 4
matriz_resultados_annealing =np.zeros((len(puntos_iniciales), len(tolerancias), ndim))

# Parámetros del Recocido Simulado
T_inicial = 2.0
enfriamiento = 0.95

print("\n" + "="*50)
print("INICIANDO SIMPLEX ANNEALING (BARRIDO MÚLTIPLE)")
print("="*50)

for i in range(len(puntos_iniciales)):
    p_inicial = puntos_iniciales[i]
    print(f"\nEvaluamos el punto inicial {p_inicial}")
    
    for j in range(len(tolerancias)):
        tolerancia = tolerancias[j]
        print(f" > Usamos la tolerancia {tolerancia}... ", end="")
        
        
        ndim = len(p_inicial)
        mpts = ndim + 1
        vertices_p = np.zeros((mpts, ndim))
        vertices_p[0] = p_inicial
        paso = 0.05
        
        for k in range(1, mpts):
            vertices_p[k] = np.copy(p_inicial)
            vertices_p[k, k-1] -= paso
            
        
        resultados_y = np.zeros(mpts)
        for k in range(mpts):
            resultados_y[k] = funcion_simplex(vertices_p[k])
            
       
        mejores_p, menor_err, iteraciones_tot = amoeba_annealing(vertices_p, resultados_y, tolerancia, funcion_simplex, T_inicial, enfriamiento)
        
       
        matriz_iteraciones_annealing[i, j] = iteraciones_tot
        matriz_errores_annealing[i, j] = menor_err
        matriz_resultados_annealing[i,j] = mejores_p
        print(f"Terminado en {iteraciones_tot} iteraciones, con error de {menor_err:.2f}. El punto obtenido es {mejores_p}")

print("\nSimplex Annealing Completado")

Se grafican los resultados de annealing:

In [ ]:
fig_comp, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
colores = ['#1f77b4', '#ff7f0e', '#2ca02c']
marcadores = ['o', 's', '^']

for i in range(len(puntos_iniciales)):
    # Gráfico de Iteraciones
    ax1.plot(tolerancias, matriz_iteraciones_annealing[i], color=colores[i], marker=marcadores[i], inestyle='-', linewidth=2, markersize=8, label=etiquetas_print[i])
    
    # Gráfico de Error Final
    ax2.plot(tolerancias, matriz_errores_annealing[i], color=colores[i], marker=marcadores[i], linestyle='-', linewidth=2, markersize=8, label=etiquetas_print[i])

# Formato Gráfico 1: Iteraciones vs Tolerancia
ax1.set_title('Convergencia (Annealing): Iteraciones requeridas', fontsize=14)
ax1.set_xlabel('Tolerancia ($ftol$)', fontsize=12)
ax1.set_ylabel('Número de Iteraciones', fontsize=12)
ax1.set_xscale('log')
ax1.grid(True, which="both", ls="--", alpha=0.5)
ax1.legend()

# Formato Gráfico 2: Error vs Tolerancia
ax2.set_title('Precisión (Annealing): Error Relativo Alcanzado', fontsize=14)
ax2.set_xlabel('Tolerancia ($ftol$)', fontsize=12)
ax2.set_ylabel('Error Relativo Mínimo', fontsize=12)
ax2.set_xscale('log')
ax2.set_yscale('log')

formateador = ScalarFormatter()
formateador.set_scientific(False)
ax2.yaxis.set_major_formatter(formateador)

ax2.grid(True, which="both", ls="--", alpha=0.5)
ax2.legend()

plt.tight_layout()
plt.show()

**Comparación de resultados**

De scipy, obtenemos las funciones equivalentes a los metodos usados anteriormente. En este caso, Nelder-Mead es para downhill simplex y dual_annealing es para annealing.

In [ ]:
ndim = 4
limites = [(0.01,5),(0.01,5),(0.01,5),(0.01,5)] #Uno por cada variable

#Para simplex de scipy
iteraciones_scipy_simplex = np.zeros((len(puntos_iniciales),len(tolerancias)))
errores_scipy_simplex = np.zeros((len(puntos_iniciales),len(tolerancias)))
vectores_scipy_simplex = np.zeros((len(puntos_iniciales),len(tolerancias),ndim))

#Para annealing de scipy
iteraciones_scipy_annealing = np.zeros((len(puntos_iniciales),len(tolerancias)))
errores_scipy_annealing = np.zeros((len(puntos_iniciales),len(tolerancias)))
vectores_scipy_annealing = np.zeros((len(puntos_iniciales),len(tolerancias),ndim))

print("Inicia Scipy")

for i in range (len(puntos_iniciales)):
    p_inicial = puntos_iniciales[i]
    print(f"Se evalua el punto inicial: {p_inicial}")
    for j in range(len(tolerancias)):
        tolerancia = tolerancias[j]
        print(f"Se usa la tolerancia {tolerancia}")
        #Se hace Nelder-Mead (downhill simplex)
        resultado_scipy_simplex = minimize(funcion_simplex,p_inicial,method="Nelder-Mead",tol=tolerancia)
        iteraciones_scipy_simplex[i,j] = resultado_scipy_simplex.nit
        errores_scipy_simplex[i,j] = resultado_scipy_simplex.fun
        vectores_scipy_simplex[i,j] = resultado_scipy_simplex.x
        print(f"En {iteraciones_scipy_simplex[i,j]} iteraciones, Simplex llega al punto {vectores_scipy_simplex[i,j]}, con un error de {errores_scipy_simplex[i,j]}")
        #Se hace dual_annealing
        resultado_scipy_annealing = dual_annealing(funcion_simplex,bounds=limites, x0=p_inicial,maxiter=10,maxfun=300) 
        #Se establecen limitaciones para agilizar el codigo
        iteraciones_scipy_annealing[i,j] = resultado_scipy_annealing.nit
        errores_scipy_annealing[i,j] = resultado_scipy_annealing.fun
        vectores_scipy_annealing[i,j] = resultado_scipy_annealing.x
        print(f"En {iteraciones_scipy_annealing[i,j]} iteraciones, Annealing llega al punto {vectores_scipy_annealing[i,j]}, con un error de {errores_scipy_annealing[i,j]}")

Graficamos todos los resultados, para ver una comparación visual.

In [ ]:
fig_comp, ejes = plt.subplots(2, 3, figsize=(18, 10))

metodos_nombres = ['Custom Simplex', 'Custom Annealing', 'Scipy Nelder-Mead', 'Scipy Annealing']

matrices_errores = [matriz_errores_simplex,matriz_errores_annealing,errores_scipy_simplex,errores_scipy_annealing]
matrices_iter = [matriz_iteraciones_simplex,matriz_iteraciones_annealing,iteraciones_scipy_simplex,iteraciones_scipy_annealing]

# 4 colores y marcadores (uno para cada método)
colores_metodos = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
marcadores_metodos = ['o', 's', '^', 'D']

for col in range(len(puntos_iniciales)): 
    ax_iter = ejes[0, col]
    ax_err = ejes[1, col]
    
    punto_actual = puntos_iniciales[col][0] 
    
    # Se recorren los 4 métodos para dibujarlos como líneas en el mismo gráfico
    for m in range(4):
        # Gráficos de Iteraciones (Fila Superior)
        ax_iter.plot(tolerancias, matrices_iter[m][col], color=colores_metodos[m], marker=marcadores_metodos[m], linestyle='-', linewidth=2, markersize=8, label=metodos_nombres[m])
        
        # Gráficos de Errores (Fila Inferior)
        ax_err.plot(tolerancias, matrices_errores[m][col], color=colores_metodos[m], marker=marcadores_metodos[m], linestyle='-', linewidth=2, markersize=8, label=metodos_nombres[m])
        
    # Formato de Iteraciones
    ax_iter.set_title(f'Iteraciones: Punto Inicial {punto_actual}', fontsize=14)
    ax_iter.set_xlabel('Tolerancia ($ftol$)', fontsize=12)
    ax_iter.set_ylabel('Iteraciones', fontsize=12)
    ax_iter.set_xscale('log')
    ax_iter.invert_xaxis()
    ax_iter.grid(True, ls="--", alpha=0.5)
    
    # Formato de Errores
    ax_err.set_title(f'Error Final: Punto Inicial {punto_actual}', fontsize=14)
    ax_err.set_xlabel('Tolerancia ($ftol$)', fontsize=12)
    ax_err.set_ylabel('Error Mínimo', fontsize=12)
    ax_err.set_xscale('log')
    ax_err.set_yscale('log')
    ax_err.invert_xaxis()
    ax_err.grid(True, ls="--", alpha=0.5)
    
    from matplotlib.ticker import ScalarFormatter
    formateador = ScalarFormatter()
    formateador.set_scientific(False)
    ax_err.yaxis.set_major_formatter(formateador)

# Se agrega la leyenda solo a la primera columna para no saturar los gráficos
ejes[0, 0].legend(fontsize=10)
ejes[1, 0].legend(fontsize=10)

plt.tight_layout()
plt.show()